# Rotation de la clé Π (permutation du vocabulaire)

La confidentialité du système repose sur la permutation Π, qui reste **côté
client** (le proxy la lit dans `artifacts/obfuscation_keys.json`). La clé
actuelle est dérivée de `seed 0` — **expérimentale et documentée** : elle ne
doit pas être utilisée telle quelle pour un usage sensible.

## Deux niveaux de rotation

| Niveau | Quoi | Protège contre | Coût |
|---|---|---|---|
| **1 — secours** | nouvelle Π aléatoire + réordonnancement des lignes embed/head du modèle servi | réutilisation d'une clé compromise | rapide (modèle local requis) |
| **2 — complète** | nouvelle seed → P̂/Q̂ + bruits + modèle re-transformé | aussi l'appariement multi-snapshots | re-transform (~1-2 h) |

Ce notebook couvre la **génération et la validation d'une nouvelle clé**
(niveau 1, partie légère), puis donne les **commandes d'application**
(partie lourde, à lancer consciemment). Référence : `docs/rotation-cles.md`.

In [ ]:
import os
# remonter jusqu'à la racine du repo (kernel nbconvert démarre dans notebooks/)
for _ in range(5):
    if os.path.isdir("artifacts") and os.path.exists(
            "artifacts/obfuscation_keys.json"):
        break
    os.chdir("..")
print("racine du repo :", os.getcwd())

## 1. Clé actuelle (lecture seule)

La clé de production actuelle (`seed 0`) — ne pas l'écraser sans avoir
préparé l'application de la rotation (étape 3).

In [ ]:
import json, os

KEYS = "artifacts/obfuscation_keys.json"
keys = json.load(open(KEYS))
perm = {int(k): int(v) for k, v in keys["vocab_permutation"].items()}
print("clé actuelle :", KEYS)
print("  seed      :", keys.get("seed"))
print("  taille    :", len(perm), "entrées (vocabulaire 151936)")
unperm_int = {int(k): int(v)
                for k, v in keys["vocab_unpermute"].items()}
print("  cohérente :", all(unperm_int[v] == k
                           for k, v in list(perm.items())[:1000]))

## 2. Générer et valider une NOUVELLE clé (aléatoire)

La nouvelle permutation est tirée au **CSPRNG** (`random.SystemRandom`) —
pas une seed reproductible. Validation : bijection + round-trip
(Π⁻¹(Π(t)) = t). La clé est écrite dans `/tmp` — pas encore appliquée.

In [ ]:
import random

def nouvelle_permutation(V=151936):
    rng = random.SystemRandom()
    p = list(range(V))
    rng.shuffle(p)
    return dict(zip(range(V), p))          # {clair: obfusqué}

# nouvelle clé de DÉMONSTRATION (dans /tmp — la production n'est pas touchée)
perm_new = nouvelle_permutation()
unperm_new = {v: k for k, v in perm_new.items()}

# validations
assert len(perm_new) == len(set(perm_new.values())) == 151936   # bijection
assert all(unperm_new[perm_new[t]] == t for t in range(10000))  # round-trip
# différente de la clé actuelle ?
diff = sum(1 for t in range(10000) if perm_new[t] != perm[t])
print(f"nouvelle permutation générée (CSPRNG) — {diff}/10000 positions "
      f"différentes de la clé actuelle")
print("round-trip Π⁻¹(Π(t)) = t : OK (10 000 ids testés)")
print("clé de démonstration NON appliquée (écrite nulle part)")

## 3. Appliquer la rotation niveau 1 (LOURD — à lancer consciemment)

Réordonne les lignes embed/head des modèles **servis** (14B et 8B) avec la
nouvelle permutation, remplace les modèles sur le volume, puis bascule la
clé et les proxies. Nécessite de **télécharger les modèles en local**
(~31 + ~17 Go). Les commandes sont données — **ne pas exécuter sans avoir
lu `docs/rotation-cles.md`** et sans avoir sauvegardé la clé actuelle.

In [ ]:
# ===== ÉTAPES LOURDES — décommenter UNIQUEMENT pour appliquer réellement =====

# 0. sauvegarde de la clé actuelle (obligatoire)
# !cp artifacts/obfuscation_keys.json artifacts/obfuscation_keys.backup.json

# 1. télécharger les modèles servis depuis le volume (une fois)
# !mkdir -p /tmp/rot && cd /tmp/rot
# !~/modal-venv/bin/modal volume get obfuscator-models qwen3-14b-h128-a1-h02 ./qwen3-14b-h128-a1-h02
# !~/modal-venv/bin/modal volume get obfuscator-models qwen3-8b-ft-h128-a1-h02 ./qwen3-8b-ft-h128-a1-h02

# 2. réordonner avec la NOUVELLE clé (générée à l'étape 2 — ici régénérée)
#    → produit des modèles indexés par Π_new + la nouvelle clé
# !~/Secretarius/Wiki_LM/.venv/bin/python tools/rotate_pi.py \
#     --model-in /tmp/rot/qwen3-14b-h128-a1-h02 \
#     --model-out /tmp/rot/qwen3-14b-h128-a1-h02-r1 \
#     --keys-in artifacts/obfuscation_keys.json \
#     --keys-out artifacts/obfuscation_keys-r1.json

# 3. remplacer sur le volume et retirer l'ancien (multi-snapshot)
# !~/modal-venv/bin/modal volume put obfuscator-models /tmp/rot/qwen3-14b-h128-a1-h02-r1 qwen3-14b-h128-a1-h02
# !~/modal-venv/bin/modal volume rm -r obfuscator-models qwen3-8b-ft-h128-a1-h02   # à refaire pour chaque modèle

# 4. basculer la clé locale (après vérification) + redémarrer les proxies
# !mv artifacts/obfuscation_keys-r1.json artifacts/obfuscation_keys.json
# !systemctl --user restart obfuscator-proxy obfuscator-proxy-8b

print("Étapes lourdes commentées — aucune modification effectuée.")

## 4. Rotation complète (niveau 2)

Pour une rotation robuste (nouveaux P̂/Q̂ + bruits + modèle re-transformé,
anciens modèles retirés), voir `docs/rotation-cles.md` — la transformation
se fait sur Modal (ou en local, 128 Go RAM) avec une nouvelle seed aléatoire
**jamais publiée**.

## Notes

- Après une rotation appliquée, le notebook `acces_tailscale.ipynb` continue
  de fonctionner **sans modification** : le proxy lit la clé au démarrage
  (penser à `systemctl --user restart obfuscator-proxy`).
- Ne **jamais** laisser l'ancienne clé ou l'ancien modèle en service (sinon
  appariement différentiel entre générations).
- La clé de production actuelle (`seed 0`) reste en place tant que la
  rotation n'est pas appliquée — ce notebook ne la modifie pas.

In [ ]:
# vérification finale : la clé de production est intacte
keys2 = json.load(open(KEYS))
print("clé de production intacte : seed", keys2.get("seed"),
      "—", len(keys2["vocab_permutation"]), "entrées")